In [ ]:
import os
import sys
import warnings
from collections import deque

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning, message=".*resized since it had shape.*")

In [ ]:
if os.path.abspath("package") not in sys.path:
    sys.path.append(os.path.abspath("package"))

In [ ]:
from package.environment import build_environment
from package.module import Rainbow, DuelingDistributionHead, BreakoutBackbone, ScaleModule
from package.actor import DistributionActor
from package.utils import build_buffer, progress, save_video_opencv
from package.td_view import to_transition, Transition
from package.optim import DistributionalLoss, SoftUpdate, DQNOptimizer

In [ ]:
import torch
import torch.nn as nn
from tensordict.nn import TensorDictModule, TensorDictSequential
from torchrl.envs import GymWrapper
from torchrl.collectors import Collector

In [ ]:
environment: GymWrapper = build_environment()
_ = environment.reset()

In [ ]:
# --- Гиперпараметры цикла обучения ---
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
TOTAL_FRAMES: int = 2_000_000  # всего кадров среды за прогон
FRAMES_PER_BATCH: int = 4  # сколько кадров собираем за одну итерацию
BATCH_SIZE: int = 32  # размер батча из буфера
UPDATES_PER_BATCH: int = 1  # градиентных шагов на один собранный батч
MIN_BUFFER: int = 20_000  # прогрев: не учимся, пока в буфере меньше переходов
BUFFER: int = 1_000_000  # Размер буфера.
LOG_INTERVAL: int = 500  # печатать статистику раз в N итераций
CKPT_PATH: str = "models/rainbow.pt"
BUFFER_DIR: str = "buffer"

In [ ]:
buf = build_buffer(size=TOTAL_FRAMES, device=DEVICE, directory=BUFFER_DIR, alpha=0.5, beta=0.4, gamma=0.99, n_steps=3)
dqn = nn.Sequential(ScaleModule(1. / 255.), Rainbow(BreakoutBackbone(), DuelingDistributionHead(4, 51)))
policy = TensorDictSequential(TensorDictModule(module=dqn, in_keys=["observation"], out_keys=["logits"]),
                              TensorDictModule(module=DistributionActor(), in_keys=["logits"], out_keys=["action"]))

In [ ]:
policy = policy.to(DEVICE)

In [ ]:
if os.path.exists(CKPT_PATH):
    dqn.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    print(f"loaded -> {CKPT_PATH}")
else:
    print("No checkpoint found.")
    _ = policy(environment.fake_tensordict().to(DEVICE))
    print("Weights initialized.")

In [ ]:
collector = Collector(build_environment,
                      frames_per_batch=FRAMES_PER_BATCH,
                      total_frames=MIN_BUFFER,
                      policy_device=DEVICE)
# -----------------------------------
for it, td in enumerate(collector, start=1):
    buf.extend(to_transition(td).td)
    progress(it, len(collector), "Filling the buffer.")

In [ ]:
loss_module = DistributionalLoss(dqn).to(DEVICE)
target_updater = SoftUpdate(loss_module, tau=5e-3)
optimizer = torch.optim.Adam(dqn.parameters(), lr=6.25e-5, eps=1.5e-4)
scheduler = None  # LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=TOTAL_FRAMES // 4)
optim = DQNOptimizer(loss_function=loss_module,
                     target_updater=target_updater,
                     optimizer=optimizer,
                     scheduler=scheduler,
                     grad_norm=10.0)

In [ ]:
collector = Collector(build_environment,
                      policy=policy,
                      frames_per_batch=FRAMES_PER_BATCH,
                      total_frames=TOTAL_FRAMES,
                      policy_device=DEVICE)
total_iters: int = len(collector)
# -----------------------------------
reward_window: deque = deque(maxlen=100)  # недавние отдачи эпизодов (в трансформированных единицах)
episode_return: float = 0.0
collected: int = 0
# -----------------------------------
for it, td in enumerate(collector, start=1):
    # 1. Новый опыт от политики -> в буфер.
    buf.extend(to_transition(td).td)
    collected += td.numel()
    # Копим отдачу эпизодов по флагам done (батч идёт в порядке времени).
    rewards = td["next", "reward"].reshape(-1).tolist()
    dones = td["next", "done"].reshape(-1).tolist()
    for r, d in zip(rewards, dones):
        episode_return += r
        if d:
            reward_window.append(episode_return)
            episode_return = 0.0
    # 2. Прогрев: пока данных мало — только собираем, не учимся.
    if len(buf) < MIN_BUFFER:  # ← ДОБАВИТЬ этот гейт
        progress(it, MIN_BUFFER // FRAMES_PER_BATCH, "Filling the buffer.")
        continue
    # 3. Несколько градиентных шагов на один собранный батч.
    losses: list[float] = []
    for _ in range(UPDATES_PER_BATCH):
        sample: Transition = to_transition(buf.sample(BATCH_SIZE))
        loss, td_errors = optim.step(sample)
        buf.update_priority(sample.raw.index, td_errors)
        losses.append(loss.item())
    # 4. Логирование.
    if it % LOG_INTERVAL == 0:
        mean_loss: float = sum(losses) / len(losses)
        mean_rew: float = sum(reward_window) / len(reward_window) if reward_window else float("nan")
        lr: float = 6.25e-5  # optim.scheduler.get_last_lr()[0]
        print(f"[{it:>4}/{total_iters}] frames={collected:>6} "
              f"loss={mean_loss:6.3f} avg_return={mean_rew:6.3f} "
              f"lr={lr:.2e} buffer={len(buf)}")
        # Сохраняем веса online-сети.
        torch.save(dqn.state_dict(), CKPT_PATH)
print(f"saved -> {CKPT_PATH}")

In [ ]:
environment.reset()
session = environment.rollout(max_steps=1000, policy=policy.to("cpu"), break_when_any_done=False)
save_video_opencv(session["observation"], fps=30)

In [3]:
50_000_000 // 32 // 100

15625